In [4]:
# ==============================================================================
# HARDIK'S NOTEBOOK: MODEL 1 - MANGANESE PROSPECTIVITY (XGBoost + Kriging)
# ==============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from pykrige.ok import OrdinaryKriging
import joblib

# 1. Load Data
df = pd.read_csv("manganese_prospectivity_data.csv")
print(f"Loaded {len(df)} prospectivity data points.")

# 2. Define Features and Target
# Using Sentinel-2 bands, mineral indices, and terrain
feature_cols = [
    'b02', 'b03', 'b04', 'b08', 'b11', 'b12',
    'ndvi', 'ferrous_index', 'clay_alteration',
    'elevation_m', 'slope_deg'
]
X = df[feature_cols]
y = df['target_occurrence']

# 3. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Train XGBoost Classifier
print("\n--- Training XGBoost Prospectivity Model ---")
clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
clf.fit(X_train, y_train)

# 5. Model Evaluation (Judges look for ROC-AUC, not just raw accuracy!)
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test, y_probs)
print(f"ROC-AUC Score: {auc_score:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Barren/Background", "Mn Occurrence"]))

# 6. Feature Importance
print("\nTop Contributing Exploration Features:")
feature_imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(feature_imp)

# 7. Kriging Spatial Interpolation on Mn Grade
# Filter to known deposit points with confirmed Mn %
deposit_points = df[df['target_occurrence'] == 1]

lons = deposit_points['longitude'].values
lats = deposit_points['latitude'].values
grades = deposit_points['mn_grade_pct'].values

print("\n--- Fitting Ordinary Kriging on Mn Ore Grade Surface ---")
OK = OrdinaryKriging(
    lons, lats, grades,
    variogram_model='spherical',
    verbose=False,
    enable_plotting=False
)
print("Kriging spatial model fitted successfully.")

# 8. Save Trained Models for Backend & Dashboard
joblib.dump(clf, "model1_prospectivity.pkl")
joblib.dump(OK, "kriging_grade_model.pkl")
print("\nSaved 'model1_prospectivity.pkl' and 'kriging_grade_model.pkl'!")

Loaded 487 prospectivity data points.

--- Training XGBoost Prospectivity Model ---
ROC-AUC Score: 0.9217

Classification Report:
                   precision    recall  f1-score   support

Barren/Background       0.87      0.83      0.85        48
    Mn Occurrence       0.85      0.88      0.86        50

         accuracy                           0.86        98
        macro avg       0.86      0.86      0.86        98
     weighted avg       0.86      0.86      0.86        98


Top Contributing Exploration Features:
b12                0.226153
b11                0.157386
ferrous_index      0.116019
elevation_m        0.085408
slope_deg          0.081657
b04                0.064976
b03                0.062320
ndvi               0.058004
clay_alteration    0.055471
b08                0.053226
b02                0.039380
dtype: float32

--- Fitting Ordinary Kriging on Mn Ore Grade Surface ---
Kriging spatial model fitted successfully.

Saved 'model1_prospectivity.pkl' and 'kriging_gr